# Powerlifting analyzer — RTMPose (cloud GPU) eval

Runs the **same pipeline** on Colab's GPU with the **RTMPose** (Apache) pose backend and
scores it against your `eval/groundtruth/` labels. No local GPU — nothing touches your PC.

**Before running:** menu **Runtime → Change runtime type → GPU (T4)**.


## 1. Check GPU + install deps


In [ ]:
!nvidia-smi -L


In [ ]:
!pip -q install rtmlib onnxruntime-gpu ultralytics opencv-python-headless numpy scipy matplotlib
import onnxruntime as ort
print('onnxruntime providers:', ort.get_available_providers())


## 2. Upload your project

On your PC, **zip the project folder** — exclude `.venv/`, `runs/`, `dataset/`, `.posecache/`
to keep it small (keep `src/`, `eval/`, `cli.py`, and `input/` with your videos).
Then run this cell and pick the zip.


In [ ]:
import glob, os, zipfile
from google.colab import files
up = files.upload()
zp = list(up)[0]
with zipfile.ZipFile(zp) as z: z.extractall('project')
cli = glob.glob('project/**/cli.py', recursive=True)[0]
PROJ = os.path.dirname(cli)
os.chdir(PROJ)
print('working dir:', os.getcwd())
print('videos:', os.listdir('input'))
print('ground truth:', os.listdir('eval/groundtruth'))


## 3. Compare all three backends (one table)

Runs **MediaPipe vs YOLO26 vs RTMPose** over your labelled clips and prints one side-by-side
scorecard. First run downloads the RTMPose + YOLO models (a few seconds).

In [ ]:
!python eval/run_eval.py --compare

## 4. (Optional) Per-clip detail for one backend

Single-backend mode prints a line per clip (predicted vs true reps, depth/lockout), handy for
seeing *which* clips a backend misses. Swap `rtmpose` for `yolo` or `mediapipe`.

In [ ]:
!python eval/run_eval.py --pose rtmpose

## 5. (Optional) Analyse one clip with RTMPose + download the annotated video

Change the filename/lift to one of your clips.


In [ ]:
!python cli.py input/deadlift-1.mov --lift deadlift --pose rtmpose
from google.colab import files
files.download('output/deadlift-1_annotated.mp4')


---
**Notes**
- RTMPose here = RTMDet (person) + RTMPose (joints) via `rtmlib`, Apache-2.0, GPU through onnxruntime (`performance` mode).
- `--compare` prints rep-count + depth/lockout accuracy for **MediaPipe vs YOLO26 vs RTMPose** in one table.
- Add more labelled clips to `eval/groundtruth/` (and videos to `input/`) before zipping to make the
  score meaningful — 2 clips isn't enough to trust.